In [1]:
import torch
from transformers import CLIPModel, CLIPProcessor
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
import pickle
from transformers import CLIPImageProcessor
from random import randint
import random
from tqdm.contrib.concurrent import thread_map

In [2]:
image_set = "utzap"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
with open(f'{image_set}/images_train.pkl', 'rb') as f:
    images = pickle.load(f)

In [5]:
def _embed_batch(model, processor, batch_images):
    """
    Worker function: preprocess a batch, run through model, normalize, return CPU tensor.
    """
    # CPU ↔ GPU pinning
    inputs = processor(images=batch_images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = model.get_image_features(**inputs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu()

In [6]:

def embed_images(
    model_name: str,
    images: list,
    batch_size: int = 32,
    num_workers: int = -1,
    backend: str = "threading",
):
    # 1) Load model & processor
    model = CLIPModel.from_pretrained(model_name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(model_name)

    # 2) Create list of image batches
    batches = [images[i : i + batch_size] for i in range(0, len(images), batch_size)]

    # 3) Run parallel with tqdm progress bar
    with tqdm_joblib(desc="Embedding batches", total=len(batches)) as progress_bar:
        results = Parallel(
            n_jobs=num_workers,
            backend=backend,
        )(
            delayed(_embed_batch)(model, processor, batch)
            for batch in batches
        )

    # 4) Concatenate results
    return torch.cat(results, dim=0)


In [7]:
image_embeddings_fashionclip = embed_images("patrickjohncyh/fashion-clip", images, batch_size=32)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding batches:   0%|          | 0/212 [00:00<?, ?it/s]

In [8]:
pickle.dump(image_embeddings_fashionclip, open(f"{image_set}/image_embeddings_fashionclip.pkl", "wb"))